# UNVERIFIED / INCOMPLETE NOTEBOOK

**未検証・途中までの参考Notebookです。** 全セルの実行成功は確認していません。

**Stopping reason:** Two fresh-T4 full-notebook validation attempts failed before scientific results: attempt 2 failed on a missing /content/results output directory (cells 10/11, fixed); attempt 3 passed cells 1-11 with zero errors but the CT-segmentation cell (12) never returned in ~149 min (suspected memory thrash near the 12.7 GB VM limit from the whole-volume EDT + label intermediates) and was stopped at ~07:26 JST 2026-10-09, well before its 4h cap. The 3-session Colab budget of this invocation is exhausted, so no further attempt was possible. No publication was attempted; GitHub unchanged. The authored notebook (with a memory-bounded chunked-EDT segmentation fix, unverified) is preserved at deliverables/p-265a57b31003291df0583dfc9ac9a57a.ipynb for an explicit resume_run_dir retry on a fresh T4.

Attempt status: failed. No manual login validation was performed. This draft may require your own authorized access, fail partway, or contain incomplete code. This attempt does not establish that the paper cannot be reproduced.

Paper: Joint 2D to 3D image registration workflow for comparing multiple slice photographs and CT scans of apple fruit with internal disorders

[PhenoPaper record](https://phenopaper.smartbreed-plant-phenotyping-platform.com/papers/p-265a57b31003291df0583dfc9ac9a57a)


# Joint 2D–3D registration of one 'Kanzi' apple — photo/CT slice alignment (minimal reproduction)

**Paper:** D.M. Schut et al., *Joint 2D to 3D image registration workflow for comparing
multiple slice photographs and CT scans of apple fruit with internal disorders*,
[arXiv:2310.01987v2](https://arxiv.org/abs/2310.01987v2) (revised 2023-12-13).

**Author code (used verbatim, fetched below):**
[`D1rk123/apple_photo_ct_workflow`](https://github.com/D1rk123/apple_photo_ct_workflow) @ `508ef0b7522b7e8c4ed728d97f3a43ce7043a63d`
with submodule [`ct_experiment_utils`](https://github.com/D1rk123/ct_experiment_utils) @ `30dd40876b44c8f61b47c2a027f94738b47efd74`.

**Data:** Zenodo [8167285](https://zenodo.org/records/8167285) (raw CT + photos, CC-BY-4.0) and
[8275793](https://zenodo.org/records/8275793) (results: metadata, photo segmentation masks,
registered CT slices, IPCED annotations). All dataset bytes are downloaded inside this notebook.

**Scope (one apple, partial demonstration).** We re-run the authors' **joint registration of one apple**
for the 5-slice subset around the annotated slice — exactly the configuration of the paper's own
subset experiment (`register_kanzi_dataset_subsets.py`: profile initialization on all usable
photos, joint `OverlapMseCost` optimization of the 5-slice subset with the authors' subset
learning-rate multiplier 500, momentum 0.75, their stopping rule). We then compare our registered
CT slice with the authors' published registered slice and compute an **IPCED-style** distance from
the authors' published annotations. This is **not** a reproduction of the paper's 107-apple
dataset-level results; the MSD-net photo segmentation (weights never published) is used through the
authors' published masks.

**実行内容(日本語):** りんご1個分のスライス写真5枚とCT体データを用い、著者の公開コードをそのまま実行して
2D→3D joint registration(プロファイル初期化 + OverlapMseCost による同時最適化)を行い、
著者の公開 registered CT slice および IPCED アノテーションと比較します。

**Runtime: GPU (T4) required.** The authors' optimization runs `device="cuda"`; on CPU the
joint cost (~17.5M sampled points/iteration) is impractically slow. In Colab select
**Runtime → Change runtime type → T4 GPU** before *Run all*. Measured on a T4: one cost+gradient
iteration ≈ 113 ms, so the authors' 100k-iteration cap corresponds to ≈ 3.1 h worst case; their
stopping rule (cost range over last 1000 iterations < 1e-5) may end it earlier. Expected total
run time ≈ 30 min – 3.5 h; downloads ≈ 1.1 GB.

**AI note.** The scientific operations are the authors' own pinned code, imported unmodified
(blob SHA-1 verified). Glue code — the `folder_locations.py` shim, the download/orchestration
cells, the plots, and an EDT-based exact-equivalent replacement of one too-slow morphology call —
is AI-written and labeled where it appears.
**AI注:** 科学計算本体は著者の pinned コードそのものです。グルーコード(ダウンロード処理・
フォルダ設定・図・一部の形態学的処理の等価実装)のみ AI が追加しています。


## 1. Environment check

The notebook requires a GPU runtime. We verify CUDA and the scientific stack
(preinstalled on Colab), then install the small `remotezip` helper used to read
members of the large Zenodo zip archives over HTTP range requests without
downloading them whole.

**日本語:** GPU ランタイムを確認し、大きな Zenodo zip の必要なメンバだけを
HTTP Range で取得するための `remotezip` をインストールします。


In [ ]:
import torch
from pathlib import Path
Path("/content/data").mkdir(exist_ok=True)
Path("/content/results").mkdir(exist_ok=True)
assert torch.cuda.is_available(), (
    "A GPU runtime is required: the paper's joint registration optimizes on CUDA. "
    "In Colab choose Runtime > Change runtime type > T4 GPU, then Run all.")
print("GPU:", torch.cuda.get_device_name(0), "| torch", torch.__version__)
import numpy, tifffile, skimage, scipy, matplotlib
print("numpy", numpy.__version__, "| tifffile", tifffile.__version__,
      "| scikit-image", skimage.__version__, "| scipy", scipy.__version__)
%pip install -q remotezip==0.12.6
import remotezip
print("remotezip ready")

## 2. Fetch the authors' pipeline code (pinned commits, SHA-verified)

We download the exact author files needed for the registration path from the two
pinned commits and verify each file's Git blob SHA-1 (same object IDs as on
GitHub). The authors' scripts expect a user-provided `folder_locations.py`; for
this notebook we provide the equivalent shim (AI-written glue). We also silence
`tqdm`'s per-iteration rendering so captured outputs stay small — a cosmetic
change only; iteration behavior is untouched.

**日本語:** 著者のパイプラインコードを pinned コミットから取得し、Git blob SHA-1 で
検証してから import します。`folder_locations.py` は本ノートブック用の置き換え
(AI 追加)、tqdm の出力抑制は見た目だけの変更です。


In [ ]:
from pathlib import Path
import requests, hashlib, sys

APCW_COMMIT = "508ef0b7522b7e8c4ed728d97f3a43ce7043a63d"  # D1rk123/apple_photo_ct_workflow
CEU_COMMIT  = "30dd40876b44c8f61b47c2a027f94738b47efd74"  # pinned submodule ct_experiment_utils

FILES = {
    "/content/apcw": [
        ("D1rk123/apple_photo_ct_workflow", APCW_COMMIT, "photo_ct_registration.py",  "d1e5b61a5372a86266a12871b358e6feecaf5b3e"),
        ("D1rk123/apple_photo_ct_workflow", APCW_COMMIT, "transformation_models.py",  "15ff3f0dbaaa2175d41e392f0a31dda19d27916f"),
        ("D1rk123/apple_photo_ct_workflow", APCW_COMMIT, "cost_functions.py",         "2435635e382363dc8daec8cac936630d70d823df"),
        ("D1rk123/apple_photo_ct_workflow", APCW_COMMIT, "image_coordinate_utils.py", "1f6dfaa4affa77e95cf7bbf94eb3cc5f1d0809b2"),
        ("D1rk123/apple_photo_ct_workflow", APCW_COMMIT, "process_apple_bboxes.py",   "5b9f58b4a84a3de12c4a58dcc5351f1664910299"),
        ("D1rk123/apple_photo_ct_workflow", APCW_COMMIT, "segment_apple_ct.py",       "5bf93004b9dcb2d2a2e5c0a397dc21d755f4bb15"),
        ("D1rk123/apple_photo_ct_workflow", APCW_COMMIT, "register_kanzi_dataset.py", "2f06427305fe19f08e88fbc86709ec6162c22dc6"),
    ],
    "/content/ct_experiment_utils": [
        ("D1rk123/ct_experiment_utils", CEU_COMMIT, "__init__.py",         "33bdee370b50228c96f1f6c5e7a9e3880770ae42"),
        ("D1rk123/ct_experiment_utils", CEU_COMMIT, "experiment_utils.py", "c6869ede543cba870149d1b2baf576a618ee3cde"),
        ("D1rk123/ct_experiment_utils", CEU_COMMIT, "tiff_handling.py",    "bbf625f7f2505384d4f44bf549501ebd86b76966"),
    ],
}

def fetch_verified(dest_dir, repo, commit, fn, blob):
    b = requests.get(f"https://raw.githubusercontent.com/{repo}/{commit}/{fn}", timeout=60).content
    digest = hashlib.sha1(b"blob " + str(len(b)).encode() + b"\x00" + b).hexdigest()
    assert digest == blob, (fn, digest, blob)
    (Path(dest_dir) / fn).write_bytes(b)
    print("verified", fn)

for dest, entries in FILES.items():
    Path(dest).mkdir(parents=True, exist_ok=True)
    for repo, commit, fn, blob in entries:
        fetch_verified(dest, repo, commit, fn, blob)

# AI-written glue: the authors ask users to write folder_locations.py themselves (see their README).
Path("/content/apcw/folder_locations.py").write_text(
    "from pathlib import Path\n"
    "def get_data_folder():\n    return Path('/content/data')\n"
    "def get_results_folder():\n    return Path('/content/results')\n")

sys.path.insert(0, "/content")
sys.path.insert(0, "/content/apcw")

# Cosmetic only: keep tqdm from rendering progress bars into captured outputs.
import tqdm
class _QuietTqdm(tqdm.tqdm):
    def display(self, *args, **kwargs):
        return
tqdm.tqdm = _QuietTqdm

import ct_experiment_utils as ceu
import register_kanzi_dataset as rk
print("author modules imported")

## 3. Resolve Zenodo file URLs and checksums

Both records are queried through their documented public record API. File
downloads later in the notebook are verified against these MD5 checksums.

**日本語:** Zenodo の公開 record API からファイル URL と MD5 を取得します。


In [ ]:
import requests

def zenodo_files(record_id):
    r = requests.get(f"https://zenodo.org/api/records/{record_id}", timeout=60)
    r.raise_for_status()
    return {f["key"]: {"url": f["links"]["self"], "size": f["size"],
                       "md5": f["checksum"].split(":", 1)[1]}
            for f in r.json()["files"]}

RAW = zenodo_files(8167285)   # raw dataset: CT scans, slice photos, browning scores (CC-BY-4.0)
RES = zenodo_files(8275793)   # results dataset: metadata, masks, registered slices, IPCED annotations
for k, v in sorted(RAW.items()):
    print(f"raw   {k:45s} {v['size']/1e9:8.2f} GB  md5:{v['md5'][:10]}…")
for k, v in sorted(RES.items()):
    print(f"res   {k:45s} {v['size']/1e6:8.2f} MB  md5:{v['md5'][:10]}…")

## 4. Download the small results-dataset archives (~18 MB, MD5-verified)

`photo metadata.zip` holds the metadata files the authors say are required to
run their code (`photo_metadata.csv`, `photo_bboxes.csv`,
`selected_point_compare_slice.csv`). `segmentation masks.zip` holds the photo
segmentation masks ("used for registration" subfolder). `results.zip` holds the
IPCED annotations. All are extracted locally under `/content`.

**日本語:** 結果データセットの小さな3つの zip を MD5 検証付きでダウンロードし、
展開します(メタデータ・写真セグメンテーションマスク・IPCED アノテーション)。


In [ ]:
import hashlib, zipfile, shutil

D = Path("/content/data"); RES_DIR = Path("/content/results_pkg")
D.mkdir(exist_ok=True); RES_DIR.mkdir(exist_ok=True)

def fetch_verified(meta, dest):
    if dest.exists() and dest.stat().st_size == meta["size"]:
        print("cached", dest.name); return
    b = requests.get(meta["url"], timeout=600).content
    assert len(b) == meta["size"], dest.name
    assert hashlib.md5(b).hexdigest() == meta["md5"], dest.name
    dest.write_bytes(b)
    print("verified", dest.name, f"{len(b)/1e6:.2f} MB")

fetch_verified(RES["photo metadata.zip"], D / "photo_metadata.zip")
fetch_verified(RES["segmentation masks.zip"], D / "segmentation_masks.zip")
fetch_verified(RES["results.zip"], D / "results.zip")

with zipfile.ZipFile(D / "photo_metadata.zip") as zf:
    zf.extractall(D)
shutil.copy(D / "photo metadata/photo_metadata.csv", D / "photo_metadata.csv")
shutil.copy(D / "photo metadata/photo_bboxes.csv", D / "photo_bboxes.csv")

with zipfile.ZipFile(D / "results.zip") as zf:
    results_names = zf.namelist()
    zf.extractall(RES_DIR)
print(len(results_names), "members in results.zip")

with zipfile.ZipFile(D / "segmentation_masks.zip") as zf:
    print("mask subfolders:", sorted({n.split('/')[1] for n in zf.namelist() if n.count('/') >= 2}))

## 5. Choose the apple and the 5-slice joint subset (deterministic)

We reuse the authors' `parse_metadata`. The apple is chosen deterministically as
the **lowest-numbered scan that is included and has an annotated slice**. The
5-slice subset is built exactly as in the authors' subset experiment
(`register_kanzi_dataset_subsets.py`): the annotation slice plus its two
neighbors on each side.

**日本語:** 著者の `parse_metadata` を使い、アノテーション付きで最も若い scan 番号の
りんごを決定論的に選び、論文の subset 実験と同じ規則で annotation slice ±2 の
5枚を joint 最適化の対象とします。


In [ ]:
photo_nrs_all = rk.parse_metadata(D / "photo_metadata.csv")
eligible = sorted(nr for nr, m in photo_nrs_all.items() if m.included and m.annotation_slice is not None)
SCAN = eligible[0]
SCAN_NAME = str(SCAN)
metadata = photo_nrs_all[SCAN]
ANN = metadata.annotation_slice
# authors' near_annotation_nrs = [ann + i for i in [0, -1, 1, -2, 2, -3]]; first five entries:
SLICES_5 = sorted(ANN + i for i in [0, -1, 1, -2, 2])
print(f"apple (scan) {SCAN}: annotation slice {ANN}, excluded slices {metadata.excluded_slices}")
print("5-slice joint subset:", SLICES_5)

## 6. Extract the apple's slice photos and published photo-segmentation masks

The cropped photos (~7 MB each) are read from `slice_photos_crop.zip` via HTTP
range requests; the masks come from the local
`segmentation masks/used for registration/` folder. They are placed in the exact
folder layout the authors' `load_photo_data` expects. All slices that have a
mask are extracted so initialization can use every usable photo, as the authors'
subset script does.

**日本語:** 写真は大きな zip から Range リクエストで、マスクはローカルの zip から
取り出し、著者の `load_photo_data` が期待するフォルダ構成に配置します。


In [ ]:
from remotezip import RemoteZip
import skimage.io

with RemoteZip(RAW["slice_photos_crop.zip"]["url"]) as z:
    with zipfile.ZipFile(D / "segmentation_masks.zip") as zf:
        mask_members = [n for n in zf.namelist()
                        if n.startswith(f"segmentation masks/used for registration/{SCAN_NAME}/")
                        and n.endswith(".png")]
    usable = sorted(int(n.rsplit("_", 1)[1].split(".")[0]) for n in mask_members)
    print("slices with published masks:", usable)
    for n in usable:
        z.extract(f"slice_photos_crop/{SCAN_NAME}/Kanzi{SCAN_NAME}_slice_{n}.png", str(D))

with zipfile.ZipFile(D / "segmentation_masks.zip") as zf:
    for n in usable:
        src = f"segmentation masks/used for registration/{SCAN_NAME}/Kanzi{SCAN_NAME}_slice_{n}.png"
        dest = D / "slice_photo_masks_crop_all" / SCAN_NAME / f"Kanzi{SCAN_NAME}_slice_{n}.png"
        dest.parent.mkdir(parents=True, exist_ok=True)
        with zf.open(src) as fsrc, open(dest, "wb") as fdst:
            shutil.copyfileobj(fsrc, fdst)

for n in usable:
    p = skimage.io.imread(D / f"slice_photos_crop/{SCAN_NAME}/Kanzi{SCAN_NAME}_slice_{n}.png")
    m = skimage.io.imread(D / f"slice_photo_masks_crop_all/{SCAN_NAME}/Kanzi{SCAN_NAME}_slice_{n}.png")
    assert p.shape[:2] == m.shape[:2], (n, p.shape, m.shape)
    print(f"slice {n}: photo {p.shape} {p.dtype}, mask {m.shape} {m.dtype}, apple fraction {float((m < 128).mean()):.3f}")

## 7. Photo crop geometry (`photo_bboxes.csv`)

`BboxDataParser` (author code) describes how each published crop sits inside the
original 1942×2590 photograph. `get_photo_coords_lists` needs it later to build
the clipping masks and the photo-grid coordinates for sampling the CT volume.

**日本語:** 著者の `BboxDataParser` で crop 幾何を読み込み、後の座標計算に備えます。


In [ ]:
bbox_data = rk.BboxDataParser(D / "photo_bboxes.csv", full_size=(1942, 2590), edge_clamp_range=50)
print("crop size (max H, max W) over the dataset:", bbox_data.crop_size)
for n in usable:
    ci = bbox_data.get_crop_info(SCAN, n)
    print(f"slice {n}: crop_min={ci.crop_min}, content extents={ci.get_crop_content_extents()}")

## 8. Download the apple's CT reconstruction from the 103 GB archive

The CT stacks are individual `.tif` files inside one huge zip. We read the
remote zip central directory once (via `remotezip`), confirm the members of
this apple are contiguous, and stream **only that byte span** (~962 MB) with
progress prints. This is the whole-archive equivalent of the authors'
`fdk_bh_corrected_recons/<scan>/` folder. (~5 min on a Colab VM.)

**日本語:** 103 GB の zip 内から対象りんごのメンバだけを Range 取得します
(約 962 MB、数分)。メンバは連続していることを確認済みです。


In [ ]:
from remotezip import RemoteZip
import time, struct

CT_URL = RAW["CT_fdk_reconstructions.zip"]["url"]
CT_PREFIX = f"CT_fdk_reconstructions/{SCAN_NAME}/"
with RemoteZip(CT_URL) as z:
    ct_infos = [z.getinfo(n) for n in z.namelist() if n.startswith(CT_PREFIX) and n.endswith(".tif")]
ct_infos.sort(key=lambda i: i.header_offset)
assert len(ct_infos) > 100, f"unexpected member count: {len(ct_infos)}"

# end of the span = end of the last member's data (local header: 30 + name + extra bytes)
h_last = ct_infos[-1].header_offset
hdr = requests.get(CT_URL, headers={"Range": f"bytes={h_last}-{h_last+29}"}, timeout=120).content
nlen, elen = struct.unpack("<HH", hdr[26:30])
SPAN_START = ct_infos[0].header_offset
SPAN_END = h_last + 30 + nlen + elen + ct_infos[-1].compress_size
print(f"{len(ct_infos)} tiff members, span {(SPAN_END-SPAN_START)/1e6:.1f} MB")

span_path = D / "ct_span.bin"
t0 = time.time(); got = 0
with requests.get(CT_URL, headers={"Range": f"bytes={SPAN_START}-{SPAN_END-1}"},
                  stream=True, timeout=1800) as resp:
    resp.raise_for_status()
    with open(span_path, "wb") as f:
        for chunk in resp.iter_content(1 << 22):
            f.write(chunk)
            prev, got = got, got + len(chunk)
            if got // (128 << 20) != prev // (128 << 20):
                print(f"  {got/1e6:.0f} MB  {time.time()-t0:.0f} s", flush=True)
assert got == SPAN_END - SPAN_START, (got, SPAN_END - SPAN_START)
print(f"downloaded {got/1e6:.1f} MB in {time.time()-t0:.0f} s")

## 9. Split the downloaded span into tiff slices and build the volume

Each member's local zip header is parsed, the deflate stream is decompressed,
and the byte length is verified against the central directory. The result is
the 3D CT volume (float32, ~1.06 GB), cached as an `.npy` file.

**日本語:** 取得したバイト列を tiff ごとに展開・検証し、CT 体積データ
(float32, 約1.06 GB) を構築します。


In [ ]:
import zlib, io
import numpy as np, tifffile

raw = span_path.read_bytes()
def member_bytes(info):
    off = info.header_offset - SPAN_START
    assert raw[off:off+4] == b"PK\x03\x04", info.filename
    nlen, elen = struct.unpack("<HH", raw[off+26:off+30])
    data_off = off + 30 + nlen + elen
    comp = raw[data_off:data_off + info.compress_size]
    return zlib.decompressobj(-15).decompress(comp) if info.compress_type == 8 else comp

probe = tifffile.imread(io.BytesIO(member_bytes(ct_infos[0])))
t0 = time.time()
vol = np.empty((len(ct_infos),) + probe.shape, dtype=np.float32)
for k, info in enumerate(ct_infos):
    arr = tifffile.imread(io.BytesIO(member_bytes(info)))
    assert arr.shape == probe.shape and arr.dtype == np.float32, (k, arr.shape, arr.dtype)
    vol[k] = arr
    if k % 100 == 0:
        print(k, "slices", flush=True)
print("volume", vol.shape, vol.dtype, f"built in {time.time()-t0:.0f} s")
np.save(D / "ct_volume.npy", vol)
del raw
print("voxel size 0.129302 mm (authors' calc_annotation_distances.py)")

## 10. Input preview — slice photographs with the published segmentation masks

The five joint-subset photographs with the authors' published photo masks
(red contour). Photographs are shown mirrored horizontally (axis=1 flip),
because the camera looks at the fruit from behind the slicing surface — this
flip is applied by the authors' `load_photo_data` before registration.

**日本語:** joint 最適化に使う5枚のスライス写真と、著者らの公開セグメンテーションマスク
(赤輪郭)を表示します。写真は装置構成上、左右反転して扱われます(著者コードの処理)。


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def load_photo_and_mask(n):
    photo = np.flip(skimage.io.imread(D / f"slice_photos_crop/{SCAN_NAME}/Kanzi{SCAN_NAME}_slice_{n}.png"), axis=1)
    mask = np.flip(skimage.io.imread(D / f"slice_photo_masks_crop_all/{SCAN_NAME}/Kanzi{SCAN_NAME}_slice_{n}.png") < 128, axis=1)
    return photo, mask

fig, axes = plt.subplots(1, len(SLICES_5), figsize=(4 * len(SLICES_5), 5))
for ax, n in zip(axes, SLICES_5):
    photo, mask = load_photo_and_mask(n)
    ax.imshow(photo)
    ax.contour(mask, levels=[0.5], colors="red", linewidths=0.8)
    ax.set_title(f"slice {n}" + (" (annotation)" if n == ANN else ""))
    ax.axis("off")
fig.suptitle(f"Apple scan {SCAN}: slice photographs + published photo segmentation masks (red)")
plt.savefig("/content/results/preview_photos.png", bbox_inches="tight", dpi=80)
plt.show()

## 11. Input preview — CT volume and its slice-area profile

Mid-volume CT slice and the CT area profile (square root of per-slice area,
the quantity the profile initialization matches against the photo masks).

**日本語:** CT 体積データの中央スライスと、スライスごとの面積プロファイルを表示します。


In [ ]:
from skimage.filters import threshold_otsu
mid = vol.shape[0] // 2
fig, axes = plt.subplots(1, 2, figsize=(12, 4.6))
axes[0].imshow(vol[mid], cmap="gray")
axes[0].set_title(f"CT slice {mid} ({vol.shape[1]}x{vol.shape[2]}, float32)")
approx = vol > threshold_otsu(vol)
axes[1].plot(np.sqrt(approx.sum(axis=(1, 2))), lw=0.8)
axes[1].set_xlabel("CT slice index"); axes[1].set_ylabel("sqrt(area) (voxels)")
axes[1].set_title("CT area profile (Otsu-thresholded)")
plt.savefig("/content/results/preview_ct.png", bbox_inches="tight", dpi=80)
plt.show()

## 12. Segment the apple shape in the CT volume (authors' algorithm)

The authors segment each scan with Otsu thresholding, a 7-voxel ball closing,
largest connected component, and hole filling (`segment_apple_ct.py`).
**Two faithful-to-source details:** their published `segment_apple` applies
`threshold_otsu(image)` per image and ignores its `threshold` argument, so
calling it on this single scan **is** the published computation; and because
skimage's `binary_closing(mask, ball(7))` is impractically slow on a
~265M-voxel volume, the closing below is computed via Euclidean distance
transforms — mathematically identical to dilation/erosion by a discrete ball
of radius 7 (behavior-preserving replacement, AI-written).

**日本語:** 著者の CT セグメンテーション(Otsu → ball(7) closing → 最大連結成分 →
穴埋め)を適用します。ball(7) の closing はユークリッド距離変換による数学的に
等価な実装に置き換えています(動作は同一)。


In [ ]:
import time
from scipy import ndimage
from skimage.measure import label

BALL_RADIUS = 7  # voxels, as in the authors' binary_closing(mask, ball(7))

def _closing_ball7_chunked(mask_bool, chunk_slices=48):
    """Closing by a discrete ball of radius 7 via Euclidean distance transforms,
    processed in z-chunks with a radius-sized halo so peak memory stays < ~300 MB
    per chunk (the whole-volume EDT peaked near the Colab VM RAM limit in a prior
    attempt). Mathematically identical to binary_closing(mask, ball(7))."""
    out = np.empty_like(mask_bool)
    Z = mask_bool.shape[0]
    for z0 in range(0, Z, chunk_slices):
        z1 = min(z0 + chunk_slices, Z)
        a, b = max(0, z0 - BALL_RADIUS), min(Z, z1 + BALL_RADIUS)
        sub = mask_bool[a:b]
        dilated = ndimage.distance_transform_edt(~sub) <= BALL_RADIUS
        out[z0:z1] = (ndimage.distance_transform_edt(dilated) > BALL_RADIUS)[z0 - a:z1 - a]
        del sub, dilated
        print(f"  closing chunk {z0}..{z1}/{Z}", flush=True)
    return out

def segment_apple(image, threshold):  # threshold argument unused, exactly as published
    t0 = time.time()
    mask = image > threshold_otsu(image)          # authors' line (threshold_otsu per image)
    print(f"  otsu threshold done in {time.time()-t0:.0f} s", flush=True)
    mask = _closing_ball7_chunked(mask)           # == binary_closing(mask, ball(7))
    print(f"  closing done in {time.time()-t0:.0f} s", flush=True)
    labels = label(mask)                          # largest connected component (authors' lines)
    assert labels.max() != 0
    mask = labels == (np.argmax(np.bincount(labels.ravel())[1:]) + 1)
    del labels
    mask = ndimage.binary_fill_holes(mask)        # fill holes (== authors' flood(mask,(0,0,0))==False)
    print(f"  largest-CC + hole filling done, total {time.time()-t0:.0f} s", flush=True)
    return mask

t0 = time.time()
ct_mask_np = segment_apple(vol, None)
print(f"segmented {ct_mask_np.shape}, apple fraction {ct_mask_np.mean():.4f}, {time.time()-t0:.0f} s")

fig, ax = plt.subplots(figsize=(6.5, 6))
ax.imshow(vol[mid], cmap="gray")
ax.contour(ct_mask_np[mid], levels=[0.5], colors="red", linewidths=0.8)
ax.set_title(f"CT slice {mid} + computed apple shape mask (red)")
plt.savefig("/content/results/ct_segmentation.png", bbox_inches="tight", dpi=80)
plt.show()

## 13. Load photos + masks through the authors' `load_photo_data`

This applies the authors' own loading path: masks are `mask < 128` and both
photos and masks are flipped along axis 1. We then index the 5-slice subset.

**日本語:** 著者の `load_photo_data`(マスクは `<128`、左右反転込み)で全ユーザー
可能写真を読み込み、5枚のサブセットを取り出します。


In [ ]:
photo_nrs, photo_masks, photos = rk.load_photo_data(
    D / "slice_photos_crop", D / "slice_photo_masks_crop_all", SCAN_NAME, metadata.excluded_slices)
print("usable photos:", photo_nrs)
assert all(n in photo_nrs for n in SLICES_5), (photo_nrs, SLICES_5)
subset_idx = [photo_nrs.index(n) for n in SLICES_5]
photos_s = [photos[i] for i in subset_idx]
photo_masks_s = [photo_masks[i] for i in subset_idx]
print("photo shape:", photos_s[0].shape, "| subset:", SLICES_5)

## 14. Initialization — area-profile matching (authors' method)

`initialize_on_profiles` fits slice thickness, z-offset and scale by minimizing
the mean squared closest-point distance between the CT and photo area profiles
(SGD with momentum 0.6, 10000 iterations — the authors' defaults in
`photo_ct_registration.py`). `initialize_xy_offsets` then sets each photo's
x/y offset from segmentation centers of mass. Initialization uses **all**
usable photos, exactly like the authors' subset script; only the subsequent
joint optimization is restricted to the 5-slice subset.

**日本語:** 著者の手法通り、面積プロファイルのマッチングで slice厚・zオフセット・
スケールを初期化し、重心合わせで各写真の x/y オフセットを初期化します。
初期化は全写真(ここでは10枚)で行うのが著者のサブセット実験の流れです。


In [ ]:
from photo_ct_registration import PhotoCtRegistration
from transformation_models import ParallelSliceSimilarityTM
import torch

ct_mask_t = torch.from_numpy(ct_mask_np)
registration = PhotoCtRegistration(ParallelSliceSimilarityTM(photo_nrs))
mse, ct_profile, photo_profile = registration.initialize_on_profiles(
    ct_mask_t, photo_masks, num_iterations=10000)
print(f"profile-init MSE {float(mse):.6f} | scale {float(registration.tm.scale):.6f} | "
      f"slice thickness {float(registration.tm.slice_thickness):.4f} voxels "
      f"({float(registration.tm.slice_thickness)*0.129302:.2f} mm)")

fig, ax = plt.subplots(figsize=(9, 4))  # visualization added for this notebook
ax.plot(ct_profile.numpy(), label="CT area profile")
pnrs = torch.tensor(photo_nrs)
ax.plot((-pnrs * registration.tm.slice_thickness + registration.tm.slice_offset).numpy(),
        (photo_profile * registration.tm.scale).numpy(), "o", ms=4,
        label="photo profile x scale (fitted mapping)")
ax.set_xlabel("CT slice index"); ax.set_ylabel("sqrt(area) (voxels)")
ax.set_title(f"Profile initialization, MSE = {float(mse):.6f}")
ax.legend()
plt.savefig("/content/results/initialization.png", bbox_inches="tight", dpi=80)
plt.show()

Path("/content/results").mkdir(exist_ok=True)
registration.initialize_xy_offsets(ct_mask_t, photo_masks)
registration.tm.write("/content/results/initialization_params.txt")

## 15. Joint optimization of all 5 slice transformations (authors' method + parameters)

`OverlapMseCost` penalizes the squared difference between the photo mask values
and the trilinearly-sampled CT mask over every clipping-mask pixel of each
photo. We optimize with the authors' subset-experiment configuration from
`register_kanzi_dataset_subsets.py`: `get_learning_rates(500, 5)`, momentum
0.75, max 100000 iterations on CUDA, stopping when the cost range over the last
1000 iterations is below 1e-5. On the probe T4 one iteration took ≈ 113 ms
(5 × 3.5M points); this cell may run for 30 min – 3.1 h depending on when the
stopping rule triggers. Cost per iteration is streamed to an error log, which
we plot afterwards.

**日本語:** 著者のサブセット実験と同一のハイパーパラメータ(学習率倍率500、momentum 0.75、
最大100000反復、直近1000反復のコスト幅<1e-5で停止)で5枚のスライス変換を
GPU 上で同時最適化します。T4 実測 ~113 ms/反復のため、最大 ~3.1 h かかる
可能性があります。


In [ ]:
import time
from cost_functions import OverlapMseCost

photo_full_coords_list, photo_edge_coords_list, \
    photo_clipping_mask_coords_list, photo_mask_values_list = \
    rk.get_photo_coords_lists(SCAN, photo_nrs, photo_masks, 10, bbox_data)

def stopping_condition(cost_log):  # the authors' stopping rule
    if len(cost_log) < 1001:
        return False
    return max(cost_log[-1000:]) - min(cost_log[-1000:]) < 0.00001

cost_function = OverlapMseCost(
    SLICES_5,
    [photo_clipping_mask_coords_list[i] for i in subset_idx],
    [photo_mask_values_list[i] for i in subset_idx],
    ct_mask_t,
)
t0 = time.time()
final_cost = registration.optimize_cost_function(
    cost_function=cost_function,
    learning_rates=rk.get_learning_rates(500, len(SLICES_5)),
    momentum=0.75,
    max_iterations=100000,
    device="cuda",
    error_log_path="/content/results/error_log_scan9.txt",
    stopping_condition=stopping_condition,
)
print(f"optimization finished in {(time.time()-t0)/60:.1f} min, final cost {float(final_cost):.8f}")
print("fitted angles (rad):", registration.tm.angles_xyz.numpy())

costs = np.loadtxt("/content/results/error_log_scan9.txt")
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(costs, lw=0.6)
ax.set_xlabel("iteration"); ax.set_ylabel("OverlapMseCost")
ax.set_title(f"Joint optimization cost ({len(costs)} iterations)")
plt.savefig("/content/results/cost_curve.png", bbox_inches="tight", dpi=80)
plt.show()

## 16. Sample the registered CT slices (authors' rendering)

Each photo's full pixel grid is mapped through the fitted transformation and
sampled from the CT volume (`volume_lookup`), reproducing the authors'
`plot_registration_result`: photo (left), registered CT slice (middle), and the
combined view (photo=green, CT=purple, right).

**日本語:** 各写真の画素グリッドを fitted 変換で CT に写像し、著者の可視化形式
(写真 / registered CT slice / 合成図)で表示します。


In [ ]:
from image_coordinate_utils import volume_lookup
from IPython.display import Image as IPyImage, display

ct_recon = torch.from_numpy(vol)
reg_slices = {}
fig_dir = Path("/content/results/registration_figures"); fig_dir.mkdir(parents=True, exist_ok=True)
for photo, photo_nr, pfc in zip(photos_s, SLICES_5,
                                [photo_full_coords_list[i] for i in subset_idx]):
    transformed = registration.tm.transform([photo_nr], [pfc])[0]
    reg_slices[photo_nr] = volume_lookup(ct_recon, transformed.cpu()).reshape(photo.shape[0:2])
    out = fig_dir / f"Kanzi{SCAN_NAME}_slice_{photo_nr}.png"
    rk.plot_registration_result(str(out), photo_nr, SCAN_NAME, photo, reg_slices[photo_nr])
    display(IPyImage(filename=str(out)))

## 17. Compare with the authors' published registered CT slice

We download the authors' registered CT slice for the annotation slice from
`registered ct slices.zip` (range request, ~13 MB) and compare it with our
registered slice **within the apple mask**: mean absolute difference and
Pearson correlation. This checks our pipeline against the paper's own output
for this apple (expected small differences from the single-scan CT
segmentation and the 5-slice subset vs their full registration).

**日本語:** 著者の公開 registered CT slice(annotation slice)を取得し、りんご
マスク内で MAE と相関を比較します。


In [ ]:
with RemoteZip(RES["registered ct slices.zip"]["url"]) as z:
    member = f"registered ct slices/{SCAN_NAME}/Kanzi{SCAN_NAME}_slice_{ANN}.tiff"
    z.extract(member, str(D / "author_results"))
auth = tifffile.imread(str(D / "author_results" / member))
ours = reg_slices[ANN].numpy()
assert auth.shape == ours.shape, (auth.shape, ours.shape)
mask_ann = photo_masks_s[SLICES_5.index(ANN)]
a = ours[mask_ann].astype(np.float64); b = auth[mask_ann].astype(np.float64)
mae = float(np.mean(np.abs(a - b)))
ncc = float(np.corrcoef(a, b)[0, 1])
print(f"within apple mask: MAE {mae:.5f} CT units | Pearson r {ncc:.4f}")

fig, axes = plt.subplots(1, 3, figsize=(17, 5))
axes[0].imshow(photos_s[SLICES_5.index(ANN)])
axes[0].set_title(f"Photograph {ANN} of apple {SCAN_NAME}")
axes[1].imshow(auth, cmap="gist_gray")
axes[1].set_title("Authors' published registered CT slice")
axes[2].imshow(ours, cmap="gist_gray")
axes[2].set_title(f"Our registered CT slice (r={ncc:.3f})")
plt.savefig("/content/results/comparison.png", bbox_inches="tight", dpi=80)
plt.show()

## 18. IPCED-style evaluation from the authors' annotations

The paper's IPCED is the average distance between core-endpoint annotations on
the photograph and on the registered CT slice, in mm
(`calc_annotation_distances.py`: `||photo_pts − ct_pts|| × scale × 0.129302`).
The CT annotations were placed on the **authors'** registered slice, so the
distances below measure the paper's photo–CT core alignment for this apple
(recomputed through the authors' formula with our fitted scale), not an
independently re-annotated evaluation of our slice. Both annotation sets are
also overlaid on our registered slice.

**日本語:** 著者が公開している core endpoint アノテーションから、論文の式
(距離 × scale × 0.129302 mm)で IPCED を再計算します。CT 側アノテーションは
著者の registered slice 上で付けられたものです。


In [ ]:
photo_pts = np.genfromtxt(RES_DIR / "results/IPCED annotations/photo IPCED annotations"
                          / f"apple_{SCAN}_photo_{ANN}.csv", delimiter=",", ndmin=2)
ct_pts = np.genfromtxt(RES_DIR / "results/IPCED annotations/CT IPCED annotations"
                       / f"apple_{SCAN}_photo_{ANN}.csv", delimiter=",", ndmin=2)
VOXEL_MM = 0.129302  # constant from the authors' calc_annotation_distances.py
scale = float(registration.tm.scale)
ipced = np.linalg.norm(photo_pts - ct_pts, axis=1) * scale * VOXEL_MM
print(f"{len(ipced)} annotated core-endpoint pairs; per-point IPCED (mm):", np.round(ipced, 3))
print(f"mean IPCED = {ipced.mean():.3f} mm (paper average over 107 apples: 1.47 ± 0.40 mm)")

cand = [n for n in results_names if "ipced" in n.lower() and n.lower().endswith(".csv")
        and "annotation" not in n.lower()]
print("per-apple metric files in results.zip:", cand)

fig, axes = plt.subplots(1, 2, figsize=(13, 6))
axes[0].imshow(photos_s[SLICES_5.index(ANN)])
axes[0].plot(photo_pts[:, 1], photo_pts[:, 0], "r+", ms=14, mew=2, label="photo annotations")
axes[0].plot(ct_pts[:, 1], ct_pts[:, 0], "cx", ms=10, mew=2, label="CT annotations (authors' slice)")
axes[0].legend(); axes[0].set_title(f"Photo {ANN} with IPCED annotations"); axes[0].axis("off")
axes[1].imshow(ours, cmap="gist_gray")
axes[1].plot(ct_pts[:, 1], ct_pts[:, 0], "cx", ms=10, mew=2, label="CT annotations")
axes[1].plot(photo_pts[:, 1], photo_pts[:, 0], "r+", ms=14, mew=2, label="photo annotations")
axes[1].legend(); axes[1].set_title(f"Our registered CT slice {ANN} + annotations"); axes[1].axis("off")
plt.savefig("/content/results/ipced_annotations.png", bbox_inches="tight", dpi=80)
plt.show()

## 19. Results summary and CSV export

Fitted transformation parameters, final cost, IPCED-style distances, and the
agreement with the authors' published slice for this apple.

**日本語:** fitted 変換パラメータ・最終コスト・IPCED・著者スライスとの一致度を
表にまとめ、CSV に保存します。


In [ ]:
import pandas as pd
summary = [
    ("apple (scan)", SCAN),
    ("joint slices", str(SLICES_5)),
    ("annotation slice", ANN),
    ("fitted scale (photo px -> CT voxel)", round(scale, 6)),
    ("fitted slice thickness (mm)", round(float(registration.tm.slice_thickness) * VOXEL_MM, 4)),
    ("fitted angles x/y/z (deg)", str(np.round(registration.tm.angles_xyz.numpy() * 180 / np.pi, 4).tolist())),
    ("final overlap MSE cost", float(f"{float(final_cost):.8f}")),
    ("optimization iterations", int(len(costs))),
    ("IPCED per-point (mm)", str(np.round(ipced, 3).tolist())),
    ("IPCED mean (mm)", round(float(ipced.mean()), 3)),
    ("ours vs authors' slice: MAE (CT units)", round(mae, 5)),
    ("ours vs authors' slice: Pearson r", round(ncc, 4)),
]
df = pd.DataFrame(summary, columns=["quantity", "value"])
df.to_csv("/content/results/minimal_run_results.csv", index=False)
df

## 20. Interpretation, scope and limitations

**What was executed.** The authors' pinned registration pipeline
(`photo_ct_registration.py`, `cost_functions.py`, `transformation_models.py`,
`image_coordinate_utils.py`, `process_apple_bboxes.py`, `ct_experiment_utils`)
ran unmodified on one apple: profile initialization over all usable photos,
then the joint 5-slice optimization with the authors' subset-experiment
hyperparameters on a T4 GPU. The registered slice for the annotation slice is
compared against the authors' published one, and IPCED is recomputed from the
published annotations with the authors' formula.

**Reading the numbers.** The overlap cost decreases and the fitted parameters
are printed above from the actual run. Agreement statistics (MAE, Pearson r,
IPCED) are computed from real outputs — see the values above; they are not
precomputed. Differences from the authors' published slice are expected from
(a) the CT mask being computed on this scan alone (the paper averaged the Otsu
threshold over all 120 scans, although the published `segment_apple` applies
Otsu per image), and (b) our 5-slice joint fit vs their full-slice fit.

**Not done here.** MSD-net photo segmentation (weights were never published; we
used the authors' published masks), beam-hardening-corrected reconstruction,
the 107-apple dataset-level evaluation, the separate-slice and disorder
comparisons. A single-apple run does not validate the paper's dataset-level
accuracy.

**参考文献 / References**
- Paper: arXiv:2310.01987v2 — Sections 2.5 (transformation model, initialization, joint optimization), 2.6.1 (IPCED), 2.7.1 (subset experiment).
- Code: D1rk123/apple_photo_ct_workflow @ 508ef0b (and submodule ct_experiment_utils @ 30dd408).
- Data: Zenodo 8167285 (CC-BY-4.0), Zenodo 8275793 (results; record states no license — used here read-only for this demonstration).
- PhenoPaper investigation p-265a57b31003291df0583dfc9ac9a57a-20261008T172538Z-1314164 was used as research guidance; all parameters above were verified against the pinned author code.
